[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/05_mechanistic_decomposition.ipynb)

In [ ]:
from pathlib import Path
import os, sys, subprocess


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


if _running_in_colab():
    repo_url = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
    repo_branch = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
    project_root = Path("/content") / "astromodel_proving"
    if not project_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_branch, repo_url, str(project_root)], check=True)
    os.chdir(project_root)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    project_root = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", ".")).resolve()
    os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Step 05 - Mechanistic decomposition of accepted cell ensembles

This notebook validates the full-scope Step 05 reviewer-response pipeline. It loads the accepted Step 04 cell-level ensemble, simulates hidden outputs with the canonical local astrocyte model, decomposes accepted fits into Kir/gap/leak and local/spatial mechanism summaries, adds the phenotype-description layer reused from the characterization notebook, and records conservative claim scope.

The implementation reuses only the missing measure vocabulary from `analysis/unified_astrocyte_K_buffering_characterization_EXECUTED_SMOKE.ipynb`: signed local/spatial flux modes, `dKs_activation_score`, available/recruited surface proxies, mode vectors, and provisional phenotype tags. It does not reuse that notebook's legacy Optuna DB loader or duplicate its ODE. Step 05 can identify candidate mechanism regimes or compensation manifolds, but predictive and perturbation support remains pending until Step 06.

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt

from src.step05_mechanistic_decomposition import Step05Config, run_step05_mechanistic_decomposition


def _env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


config = Step05Config(
    max_candidates=None,
    time_points=_env_int("ASTROMODEL_STEP05_TIME_POINTS", 120),
    bootstrap_iterations=_env_int("ASTROMODEL_STEP05_BOOTSTRAP_ITERATIONS", 8),
    random_seed=13,
    min_candidates_for_multicluster=6,
    min_cells_for_multicluster=3,
    min_cells_per_cluster=2,
    proxy_failure_downgrade_fraction=0.5,
    write_outputs=True,
)
result = run_step05_mechanistic_decomposition(project_root, config)
out_dir = project_root / "outputs" / "mechanisms"
print(json.dumps(result["analysis_summary"], indent=2))


## 1. Accepted ensemble inventory

In [ ]:
clusters = result["mechanism_clusters"]
flux = result["accepted_fit_mechanisms"]
inventory = clusters.groupby(["region", "condition"], as_index=False).agg(
    n_cells=("file_id", "nunique"),
    n_candidates=("candidate_id", "nunique"),
    n_clusters=("mechanism_cluster", "nunique"),
)
inventory

## 2. Candidate-sweep flux decomposition table

In [ ]:
flux_cols = [
    "file_id", "region", "condition", "candidate_id", "sweep", "current_na", "simulation_status",
    "I_Kir_integral", "I_kgap_integral", "I_leak_integral", "gap_to_kir_integral_ratio",
    "gap_fraction", "kir_fraction", "leak_fraction", "K_o_peak", "K_o_recovery_error", "proxy_validity_class",
]
flux[flux_cols].head(12)

## 2b. Protocol and K bath audit

Step 05 must not collapse the six current protocol into repeated 100 nA bath-drive simulations. The model already knows the historical current-specific bath values; Step 05 only applies explicit per-sweep/per-current/affine/gain overrides when they are present.


In [ ]:
protocol_cols = [
    "candidate_id",
    "current_na",
    "sweep",
    "K_bath_middle_used",
    "K_bath_override_mode",
    "stim_window_start_s",
    "stim_window_end_s",
    "K_o_peak",
    "K_o_recovery_error",
]
protocol_audit = flux[protocol_cols].sort_values(["candidate_id", "current_na"])
protocol_summary = flux.groupby("candidate_id", as_index=False).agg(
    n_currents=("current_na", "nunique"),
    n_unique_kbath=("K_bath_middle_used", "nunique"),
    min_kbath=("K_bath_middle_used", "min"),
    max_kbath=("K_bath_middle_used", "max"),
    override_modes=("K_bath_override_mode", lambda s: ";".join(sorted(set(map(str, s))))),
)
protocol_summary


## 3. Mechanism-space clustering

In [ ]:
cluster_evidence = clusters[[
    "file_id", "candidate_id", "mechanism_cluster", "cluster_evidence_status",
    "cluster_evidence_reason", "cluster_claim_scope", "proxy_validity_status",
]].drop_duplicates()
cluster_evidence

fig, ax = plt.subplots(figsize=(6, 4))
for label, sub in clusters.groupby("mechanism_cluster"):
    ax.scatter(sub["gap_fraction_mean"], sub["kir_fraction_mean"], s=80, label=label)
for _, row in clusters.iterrows():
    ax.annotate(str(row["candidate_id"]).split("__")[-1], (row["gap_fraction_mean"], row["kir_fraction_mean"]), fontsize=8)
ax.set_xlabel("Mean gap fraction")
ax.set_ylabel("Mean Kir fraction")
evidence_status = str(clusters["cluster_evidence_status"].iloc[0]) if "cluster_evidence_status" in clusters else "unknown"
ax.set_title(f"Step 05 mechanism-space cluster view ({evidence_status})")
ax.legend(title="Cluster")
fig.tight_layout()
plt.show()


## 4. Flux decomposition for representative candidates

In [ ]:
reps = result["representatives"]
rep_flux = flux.merge(reps[["file_id", "candidate_id", "representative_rank"]], on=["file_id", "candidate_id"], how="inner")
fig, ax = plt.subplots(figsize=(7, 4))
for (rank, candidate_id), sub in rep_flux.groupby(["representative_rank", "candidate_id"]):
    sub = sub.sort_values("current_na")
    ax.plot(sub["current_na"], sub["gap_fraction"], marker="o", label=f"rep {rank} gap")
    ax.plot(sub["current_na"], sub["kir_fraction"], marker="s", linestyle="--", label=f"rep {rank} Kir")
ax.set_xlabel("Current (nA)")
ax.set_ylabel("Flux fraction")
ax.set_ylim(0, 1)
ax.set_title("Representative Kir/gap decomposition across sweeps")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()
reps[["representative_rank", "file_id", "condition", "mechanism_cluster", "dominant_mechanism", "claim_scope"]]

## 5. Compensation-vs-separated-mode interpolation diagnostics

In [ ]:
geometry = result["geometry_classification"]
geometry[["cluster_a", "cluster_b", "geometry_screen_type", "interpolation_status", "geometry_classification", "validation_status"]]


## 6. Bootstrap stability table

In [ ]:
stability = result["bootstrap_cluster_stability"]
stability


## 7. DH/VH and condition occupancy table

In [ ]:
enrichment = result["region_mechanism_enrichment"]
enrichment.sort_values(["region", "condition", "mechanism_cluster"])

## 8. Windowed phenotype-description layer

The table below is descriptive and provisional. It uses accepted Step 04 ensembles and the canonical simulator, then adds the characterization notebook's missing local/spatial and recruitment measures. These tags do not become phenotype claims unless Step 06 supports prediction and perturbation robustness.

In [ ]:
measure_registry = result["measure_registry_status"]
windowed = result["accepted_fit_mechanisms_windowed"]
phenotypes = result["buffering_phenotype_tags"]
mode_vectors = result["M_mode_vector_by_configuration"]

display(measure_registry)
display(phenotypes.head(20))
display(mode_vectors.head(12))

## 9. Phenotype counts by region, condition, and window

In [ ]:
phenotype_counts = result["phenotype_counts_by_region_condition_window"]
phenotype_counts.sort_values(["region", "condition", "window", "buffering_phenotype"]).head(40)

## 10. Explicit claim-scope table

In [ ]:
claims = result["claim_scope_table"]
claims[[
    "claim_topic", "allowed_pre_step06_claim", "forbidden_pre_step06_claim",
    "required_next_step", "n_clusters", "n_independent_cells", "n_candidates",
    "cluster_evidence_status", "proxy_validity_status",
]]


In [ ]:
assert (flux["simulation_status"] == "ok").any()
assert {"region", "condition", "mechanism_cluster", "cluster_evidence_status"}.issubset(clusters.columns)
assert "K_bath_middle_used" in flux.columns
assert flux.groupby("candidate_id")["K_bath_middle_used"].nunique().min() > 1
assert not windowed.empty
assert windowed["dKs_activation_score"].between(0.0, 1.0).all()
assert "buffering_phenotype" in phenotypes.columns
assert claims["forbidden_pre_step06_claim"].str.contains("candidate_degenerate_regimes").any()
assert not claims["allowed_pre_step06_claim"].str.contains("candidate_degenerate_regimes").any()
if clusters["cluster_evidence_status"].eq("insufficient_evidence").any():
    assert claims.loc[claims["claim_topic"] == "mechanism_diversity", "allowed_pre_step06_claim"].iloc[0] == "insufficient_evidence_for_candidate_mechanism_regimes"
print(f"Step 05 full-scope notebook validated and outputs saved under {out_dir}")


## Post-execution scientific status

Executed status for reviewer response: Step 05 ran on the full 2110 accepted-candidate ensemble and produced 12,660 candidate-sweep rows, 50,572 windowed mechanism rows, 2110 phenotype-tag rows, and 3 candidate mechanism clusters across 35 independent accepted cells. This supports R5 model characterization and a candidate-regime screen, including the reused local/spatial/recruitment vocabulary from the characterization notebook. The maturity level remains `candidate_regime_screen`: phenotype and mechanism labels are descriptive until Step 06 predictive/perturbation support and later guardrails mature.